In [2]:
import os
from pathlib import Path
from collections import defaultdict
import numpy as np
from tqdm import tqdm, trange
from einops import einsum
import cv2
from PIL import Image
import imageio.v3 as iio
import lovely_tensors as lt
lt.monkey_patch()
import torch
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader

from hmr4d.dataset.bedlam.bedlam import BedlamDatasetV2
from hmr4d.dataset.bedlam.utils import mid2vname
from hmr4d.datamodule.mocap_trainX_testY import collate_fn

from hmr4d.utils.geo_transform import apply_T_on_points, compute_T_ayfz2ay
from hmr4d.utils.geo.hmr_cam import create_camera_sensor
from hmr4d.utils.vis.cv2_utils import draw_bbx_xys_on_image_batch, draw_kpts_with_conf_batch
from hmr4d.utils.vis.renderer import Renderer, get_global_cameras_static, get_ground_params_from_points
from hmr4d.utils.body_model import BodyModelSMPLX, BodyModelSMPLH
from hmr4d.utils.body_model.smplx_lite import SmplxLiteV437Coco17

os.environ.setdefault('CUDA_VISIBLE_DEVICES', '3')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device =', device)


device = cuda


In [3]:
smpl = BodyModelSMPLH(
    model_path="inputs/checkpoints/body_models", model_type="smpl",
    gender="neutral", num_betas=10, create_body_pose=False, 
    create_betas=False, create_global_orient=False, create_transl=False,
).cuda()
smplx = BodyModelSMPLX(
    model_path="inputs/checkpoints/body_models", model_type="smplx",
    gender="neutral", num_pca_comps=12, flat_hand_mean=False,
).cuda()
smplx_coco = SmplxLiteV437Coco17().cuda()
smplx2smpl = torch.load("hmr4d/utils/body_model/smplx2smpl_sparse.pt").cuda()
faces_smpl = smpl.faces
J_regressor = torch.load("hmr4d/utils/body_model/smpl_neutral_J_regressor.pt").cuda()

/tmp/ipykernel_1070587/465735466.py:11: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  smplx2smpl = torch.load("hmr4d/utils/body_model/smplx2smpl_sparse.pt").cuda()
/tmp/ipyk

In [4]:
ds=BedlamDatasetV2(load_image=True, load_indices=[0, -1], end_frame_window=10)

[02/16 13:51:47][INFO] [BEDLAM] Loading from inputs/BEDLAM/hmr4d_support
[02/16 13:51:48][INFO] [BEDLAM] Start loading motion files
[02/16 13:52:36][INFO] [BEDLAM] Motion files loaded. Elapsed: 49.00s
[02/16 13:52:36][INFO] [BEDLAM] 37537 sequences. 


In [ ]:
# mid = 'inputs/bedlam/bedlam_download/20221010_3-10_500_batch01hand_zoom_suburb_d/mp4/seq_000001.mp4-rp_henry_posed_001'


for idx in range(30000, 37537):
    mid = ds.idx2meta[idx]
    range1, range2 = ds.mid_to_valid_range[mid]
    # print(range1, range2)
    saved = ds._get_saved_frame_indices(mid)
    saved = sorted({int(f) for f in saved if range1 <= int(f) < range2})
    mlength = range2 - range1

    min_len_eff = min(ds.min_motion_frames, mlength)
    max_len_eff = min(ds.max_motion_frames, mlength)

    # print("saved frames:", saved)
    # print(min_len_eff, max_len_eff)

    start_candidates = sorted([f for f in saved if range1 <= f < range1 + ds.end_frame_window])
    end_candidates = [f for f in saved if start_candidates[0]+min_len_eff <= f < start_candidates[-1]+max_len_eff]
    # print("start_candidates:", start_candidates)
    # print("end_candidates:", end_candidates)    

    if len(end_candidates) == 0:
        video_path = ds.video_root / mid2vname(mid)
        out_dir = video_path.parent.parent / "vis" / video_path.stem
        print(idx, start_candidates[0]+min_len_eff, start_candidates[-1]+max_len_eff, "No end candidates. Video path:", video_path)
        for frame_idx in range(start_candidates[0]+min_len_eff, start_candidates[-1]+max_len_eff):
            try:
                frame = iio.imread(video_path, index=int(frame_idx), plugin="pyav")
                out_path = out_dir / f"{int(frame_idx):05d}.jpg"
                iio.imwrite(out_path, frame)
            except Exception as e:
                pass

{'meta': {'data_name': 'bedlam',
  'idx': 250,
  'vid': 'inputs/bedlam/bedlam_download/20221010_3-10_500_batch01hand_zoom_suburb_d/mp4/seq_000042.mp4-rp_caren_posed_008',
  'start_end': (3, 99)},
 'length': 96,
 'smpl_params_c': {'body_pose': tensor[120, 63] n=7560 (30Kb) x∈[-1.166, 1.483] μ=-0.006 σ=0.368,
  'betas': tensor[120, 10] n=1200 (4.7Kb) x∈[-1.108, 1.013] μ=0.099 σ=0.563,
  'transl': tensor[120, 3] n=360 (1.4Kb) x∈[-0.222, 11.973] μ=4.048 σ=5.421,
  'global_orient': tensor[120, 3] n=360 (1.4Kb) x∈[-3.133, 3.126] μ=-0.213 σ=1.743},
 'smpl_params_w': {'body_pose': tensor[120, 63] n=7560 (30Kb) x∈[-1.166, 1.483] μ=-0.006 σ=0.368,
  'betas': tensor[120, 10] n=1200 (4.7Kb) x∈[-1.108, 1.013] μ=0.099 σ=0.563,
  'transl': tensor[120, 3] n=360 (1.4Kb) x∈[-0.711, 1.238] μ=0.377 σ=0.684,
  'global_orient': tensor[120, 3] n=360 (1.4Kb) x∈[-3.141, 3.121] μ=-0.067 σ=1.475},
 'R_c2gv': tensor[120, 3, 3] n=1080 (4.2Kb) x∈[-0.118, 0.999] μ=0.332 σ=0.472,
 'gravity_vec': tensor[3] x∈[-1.000, 

In [ ]:
idx = 260
mid = ds.idx2meta[idx]
# [i for i,x in enumerate(ds.idx2meta) if x == "inputs/bedlam/bedlam_download/20221010_3-10_500_batch01hand_zoom_suburb_d/mp4/seq_000043.mp4-rp_ben_posed_004"]
# mid = "inputs/bedlam/bedlam_download/20221010_3-10_500_batch01hand_zoom_suburb_d/mp4/seq_000064.mp4-rp_eve_posed_003"
range1, range2 = ds.mid_to_valid_range[mid]
print(range1, range2)
saved = ds._get_saved_frame_indices(mid)
saved = sorted({int(f) for f in saved if range1 <= int(f) < range2})
mlength = range2 - range1

min_len_eff = min(ds.min_motion_frames, mlength)
max_len_eff = min(ds.max_motion_frames, mlength)

print("saved frames:", saved)
print(min_len_eff, max_len_eff)

start_candidates = sorted([f for f in saved if range1 <= f < range1 + ds.end_frame_window])
end_candidates = [f for f in saved if start_candidates[0]+min_len_eff-1 <= f < start_candidates[-1]+max_len_eff]
print("start_candidates:", start_candidates)
print("end_candidates:", end_candidates)    

# if len(end_candidates) == 0:
#     video_path = ds.video_root / mid2vname(mid)
#     out_dir = video_path.parent.parent / "vis" / video_path.stem
#     print(idx, start_candidates[0]+min_len_eff, start_candidates[-1]+max_len_eff, "No end candidates. Video path:", video_path)
#     for frame_idx in range(start_candidates[0]+min_len_eff, start_candidates[-1]+max_len_eff):
#         try:
#             frame = iio.imread(video_path, index=int(frame_idx), plugin="pyav")
#             out_path = out_dir / f"{int(frame_idx):05d}.jpg"
#             iio.imwrite(out_path, frame)
#         except Exception as e:
#             pass

In [ ]:

for i in range(5000,10000):
    batch = ds[i]
    pass
    # batch = ds[i]
    # if batch['image'].shape[0] == 0:
    #     continue
    # print(f"Batch {i}: Image : {batch['image'].shape} CamAng : {batch['cam_angvel'].shape}")

In [ ]:
window = 10
loader = DataLoader(
    ds,
    batch_size=64,
    shuffle=False,
    num_workers=0,
    drop_last=False,
    collate_fn=collate_fn,
)

seen = 0
for bi, batch in enumerate(loader):
    print(f"{bi}: batch_size={batch['image'].shape}, cam_angvel_shape={batch['cam_angvel'].shape}")
    if batch['cam_angvel'].shape[1] != 120:
        print(f"Batch {bi} has cam_angvel with shape {batch['cam_angvel'].shape}, expected 120 frames.")
    seen += 1
# counter = 0
# for idx in range(len(ds)):
#     mid = ds.idx2meta[idx]
#     data = ds.motion_files[mid].copy()
#     range1, range2 = ds.mid_to_valid_range[mid]
#     saved = ds._get_saved_frame_indices(mid)
#     saved = sorted({int(f) for f in saved if range1 <= int(f) < range2})
#     # start_missing = [x for x in range(range1, range1 + window) if x not in saved]
#     end_missing = [x for x in range(range2-window, range2) if x not in saved]
#     # if len(start_missing) > 0:
#     #     print(f"Start missing frames for idx={idx}, range1={range1}, range2={range2}, saved_len={len(saved)}, missing={start_missing}")
#     if len(end_missing) > 0:
#         print(f"End missing frames for idx={idx}, range1={range1}, range2={range2}, saved_len={len(saved)}, missing={end_missing}")
#         counter += 1
# print(f"Total {counter} samples with missing end frames within window={window}")